In [ ]:
# Install PyTorch Geometric
!pip install -q torch-geometric torch-scatter torch-sparse

print("✓ Dependencies installed successfully")

## 2. Import Libraries

In [10]:
import os
import sys
import json
import shutil
import torch
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from torch_geometric.utils import k_hop_subgraph
from datasets import load_dataset
from tqdm import tqdm
import torch_geometric
from collections import defaultdict

# Check environment
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.9.1+cpu
CUDA available: False


## 3. Configuration

Adjust these parameters if needed.

In [11]:
# Configuration
CONFIG = {
    'output_dir': '/kaggle/working/fb15k237_gwm_data',
    'bert_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'num_hops': 5,
    'sample_size': 5,  # Max neighbors to sample per hop
    'embedding_dim': 2048,  # Target dimension for GWM
    'random_state': 42,  # For reproducibility
    'batch_size': 32,  # BERT encoding batch size
    'max_train_samples': None,  # Use all available data (dataset already has splits)
}

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG['device'] = device

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  output_dir: /kaggle/working/fb15k237_gwm_data
  bert_model: sentence-transformers/all-MiniLM-L6-v2
  num_hops: 5
  sample_size: 5
  embedding_dim: 2048
  random_state: 42
  batch_size: 32
  max_train_samples: None
  device: cpu


## 4. Download Raw FB15k-237 Dataset

Download from HuggingFace Hub.

In [12]:
print("="*70)
print(" "*20 + "STEP 1: Loading FB15k-237 Dataset")
print("="*70)

print("Loading dataset from KGraph/FB15k-237...")

dataset = load_dataset("KGraph/FB15k-237")

print(f"\n✓ Successfully loaded FB15k-237 dataset!")
print(f"  Train samples: {len(dataset['train']):,}")
print(f"  Validation samples: {len(dataset['validation']):,}")
print(f"  Test samples: {len(dataset['test']):,}")

# Explore the dataset structure
print("\nDataset structure:")
print(f"  Splits: {list(dataset.keys())}")
print(f"  Features: {dataset['train'].features}")

# Show example
example = dataset['train'][0]
print(f"\nExample triple:")
for key, value in example.items():
    print(f"  {key}: {value}")

                    STEP 1: Loading FB15k-237 Dataset
Loading dataset from KGraph/FB15k-237...

✓ Successfully loaded FB15k-237 dataset!
  Train samples: 272,115
  Validation samples: 17,535
  Test samples: 20,466

Dataset structure:
  Splits: ['train', 'validation', 'test']
  Features: {'text': Value('string')}

Example triple:
  text: /m/027rn	/location/country/form_of_government	/m/06cx9


In [18]:
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
!hf auth login --token {HF_TOKEN}

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `GWM` has been saved to C:\Users\Modern\.cache\huggingface\stored_tokens
Your token has been saved to C:\Users\Modern\.cache\huggingface\token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [21]:
# Extract all unique entities and relations
print("\nExtracting entities and relations...")
all_entities = set()
all_relations = set()

for split in ['train', 'validation', 'test']:
    for item in dataset[split]:
        item_splits = item['text'].split('\t')
        all_entities.add(item_splits[0])
        all_entities.add(item_splits[2])
        all_relations.add(item_splits[1])

entities = sorted(list(all_entities))
relations = sorted(list(all_relations))

# Create mappings
entity_to_id = {ent: idx for idx, ent in enumerate(entities)}
relation_to_id = {rel: idx for idx, rel in enumerate(relations)}

print(f"\n✓ Extracted:")
print(f"  Total entities: {len(entities):,}")
print(f"  Total relations: {len(relations):,}")

# Load entity name and description mappings
print("\n" + "="*70)
print("Loading entity name and description mappings...")
print("="*70)

# Download mapping files from KGraph/FB15k-237 raw folder
from huggingface_hub import hf_hub_download

mid2name_file = hf_hub_download(
    repo_id="KGraph/FB15k-237",
    filename="data/FB15k_mid2name.txt",
    repo_type="dataset"
)

mid2desc_file = hf_hub_download(
    repo_id="KGraph/FB15k-237", 
    filename="data/FB15k_mid2description.txt",
    repo_type="dataset"
)

# Load name mappings
entity_to_name = {}
with open(mid2name_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            mid, name = parts
            entity_to_name[mid] = name.replace('_', ' ')

print(f"✓ Loaded {len(entity_to_name):,} entity names")

# Load description mappings
entity_to_desc = {}
with open(mid2desc_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            mid, desc = parts
            # Remove quotes and language tag (@en)
            desc = desc.strip('"').split('"@')[0]
            entity_to_desc[mid] = desc

print(f"✓ Loaded {len(entity_to_desc):,} entity descriptions")

# Show examples
print(f"\nExample mappings:")
example_entities = list(entities)[:10]
for ent in example_entities:
    name = entity_to_name.get(ent, "N/A")
    desc = entity_to_desc.get(ent, "N/A")
    desc_preview = desc[:100] + "..." if len(desc) > 100 else desc
    print(f"\n  MID: {ent}")
    print(f"  Name: {name}")
    print(f"  Description: {desc_preview}")


Extracting entities and relations...

✓ Extracted:
  Total entities: 14,541
  Total relations: 237

Loading entity name and description mappings...



Extracting entities and relations...

✓ Extracted:
  Total entities: 14,541
  Total relations: 237

Loading entity name and description mappings...


FB15k_mid2name.txt: 0.00B [00:00, ?B/s]


Extracting entities and relations...

✓ Extracted:
  Total entities: 14,541
  Total relations: 237

Loading entity name and description mappings...


FB15k_mid2name.txt: 0.00B [00:00, ?B/s]

data/FB15k_mid2description.txt:   0%|          | 0.00/13.2M [00:00<?, ?B/s]


Extracting entities and relations...

✓ Extracted:
  Total entities: 14,541
  Total relations: 237

Loading entity name and description mappings...


FB15k_mid2name.txt: 0.00B [00:00, ?B/s]

data/FB15k_mid2description.txt:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

✓ Loaded 14,951 entity names
✓ Loaded 14,904 entity descriptions

Example mappings:

  MID: /m/010016
  Name: Denton
  Description: Denton is a city in the U.S. state of Texas and the county seat of Denton County. As of the 2010 Uni...

  MID: /m/0100mt
  Name: El Paso
  Description: El Paso is the county seat of El Paso County, Texas, United States, and lies in far West Texas. As o...

  MID: /m/0102t4
  Name: Marshall
  Description: Marshall is a city in and the county seat of Harrison County in the northeastern corner of Texas. Ma...

  MID: /m/0104lr
  Name: Beaumont
  Description: Beaumont is a city in and county seat of Jefferson County, Texas, United States, within the Beaumont...

  MID: /m/0105y2
  Name: Lubbock
  Description: Lubbock is a city in and the county seat of Lubbock County, Texas, United States. The city is locate...

  MID: /m/0106dv
  Name: Waco
  Description: Waco is a city in and the county seat of McLennan County, Texas, United States. It is situated along...


## 5. Create Rich Entity Texts and Generate BERT Embeddings

Use entity names + descriptions for much better embeddings.

In [23]:
print("="*70)
print(" "*20 + "STEP 2: Creating Entity Texts & BERT Embeddings")
print("="*70)

# Create rich entity texts: Name + Description (if available)
print("Creating rich entity texts from names and descriptions...")
entity_texts = []
entities_with_desc = 0
entities_with_name = 0
entities_fallback = 0

for ent in entities:
    name = entity_to_name.get(ent, None)
    desc = entity_to_desc.get(ent, None)
    
    if name and desc:
        # Best case: Name + Description
        text = f"Entity: {name}. Description: {desc}"
        entities_with_desc += 1
    elif name:
        # Good case: Just name
        text = f"Entity: {name}"
        entities_with_name += 1
    else:
        # Fallback: Convert MID to readable text
        text = ent.replace('/m/', '').replace('_', ' ')
        entities_fallback += 1
    
    entity_texts.append(text)

print(f"\nEntity text statistics:")
print(f"  With name + description: {entities_with_desc:,} ({entities_with_desc/len(entities)*100:.1f}%)")
print(f"  With name only: {entities_with_name:,} ({entities_with_name/len(entities)*100:.1f}%)")
print(f"  Fallback (MID): {entities_fallback:,} ({entities_fallback/len(entities)*100:.1f}%)")

print(f"\nExample entity texts:")
for i in range(min(3, len(entities))):
    text_preview = entity_texts[i][:200] + "..." if len(entity_texts[i]) > 200 else entity_texts[i]
    print(f"\n  [{i}] {entities[i]}")
    print(f"      {text_preview}")

# Generate BERT embeddings
print(f"\n{'='*70}")
print(f"Loading BERT model: {CONFIG['bert_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['bert_model'])
bert_model = AutoModel.from_pretrained(CONFIG['bert_model']).to(device)
bert_model.eval()

print(f"✓ Model loaded on {device}")

# Generate embeddings with batching
all_embeddings = []
batch_size = CONFIG['batch_size']

print(f"\nEncoding {len(entity_texts):,} entity texts (batch_size={batch_size})...")
print("This will take a while due to long descriptions...")

with torch.no_grad():
    for i in tqdm(range(0, len(entity_texts), batch_size), desc="BERT encoding"):
        batch_texts = entity_texts[i:i+batch_size]
        
        # Tokenize with longer max_length for descriptions
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,  # Use full context for descriptions
            return_tensors='pt'
        ).to(device)
        
        # Get embeddings (mean pooling)
        outputs = bert_model(**encoded)
        embeddings = outputs.last_hidden_state.mean(dim=1)
        all_embeddings.append(embeddings.cpu())

node_embeddings = torch.cat(all_embeddings, dim=0)
print(f"\n✓ Generated embeddings shape: {node_embeddings.shape}")

# Pad/truncate to target dimension
current_dim = node_embeddings.shape[1]
target_dim = CONFIG['embedding_dim']

if current_dim < target_dim:
    padding = torch.zeros(node_embeddings.shape[0], target_dim - current_dim)
    node_embeddings = torch.cat([node_embeddings, padding], dim=1)
    print(f"  Padded from {current_dim} to {target_dim} dimensions")
elif current_dim > target_dim:
    node_embeddings = node_embeddings[:, :target_dim]
    print(f"  Truncated from {current_dim} to {target_dim} dimensions")

print(f"\n✓ Final embeddings shape: {node_embeddings.shape}")

# Free GPU memory
del bert_model, tokenizer
if device == 'cuda':
    torch.cuda.empty_cache()
    print(f"✓ Freed GPU memory")

                    STEP 2: Creating Entity Texts & BERT Embeddings
Creating rich entity texts from names and descriptions...

Entity text statistics:
  With name + description: 14,515 (99.8%)
  With name only: 26 (0.2%)
  Fallback (MID): 0 (0.0%)

Example entity texts:

  [0] /m/010016
      Entity: Denton. Description: Denton is a city in the U.S. state of Texas and the county seat of Denton County. As of the 2010 United States Census, its population was 113,383, making it the 27th most ...

  [1] /m/0100mt
      Entity: El Paso. Description: El Paso is the county seat of El Paso County, Texas, United States, and lies in far West Texas. As of July 1, 2012, the population estimate from the U.S. Census was 672,5...

  [2] /m/0102t4
      Entity: Marshall. Description: Marshall is a city in and the county seat of Harrison County in the northeastern corner of Texas. Marshall is a major cultural and educational center in East Texas and t...


## 6. Build Graph Structure and Create Multi-hop Embeddings

Build edge_index from triples and aggregate k-hop neighborhoods.

In [24]:
print("="*70)
print(" "*20 + "STEP 3: Building Graph & Multi-hop Embeddings")
print("="*70)

# Build edge_index from all triples (train + validation + test)
print("Building graph structure from all triples...")
edge_list = []

for split in ['train', 'validation', 'test']:
    for item in tqdm(dataset[split], desc=f"Processing {split}", leave=False):
        item_splits = item['text'].split('\t')
        head_id = entity_to_id[item_splits[0]]
        tail_id = entity_to_id[item_splits[2]]
        edge_list.append([head_id, tail_id])

edge_index = torch.tensor(edge_list, dtype=torch.long).t()
print(f"\n✓ Graph structure:")
print(f"  Nodes: {len(entities):,}")
print(f"  Edges: {edge_index.size(1):,}")

# Create multi-hop embeddings
num_nodes = len(entities)
embedding_dim = node_embeddings.shape[1]
num_hops = CONFIG['num_hops']
sample_size = CONFIG['sample_size']

# Initialize multi-hop embeddings
multi_hop_embs = torch.zeros(num_hops, num_nodes, embedding_dim)

print(f"\nBuilding {num_hops}-hop neighborhoods for {num_nodes:,} nodes...")
print(f"Sampling up to {sample_size} neighbors per hop\n")

# Set random seed for reproducible neighbor sampling
np.random.seed(CONFIG['random_state'])

for node_idx in tqdm(range(num_nodes), desc="Creating multi-hop embeddings"):
    for hop in range(num_hops):
        if hop == 0:
            # Hop 0: Use the node's own embedding
            multi_hop_embs[hop, node_idx] = node_embeddings[node_idx]
        else:
            # Get k-hop subgraph
            subset, _, _, _ = k_hop_subgraph(
                node_idx=node_idx,
                num_hops=hop,
                edge_index=edge_index,
                relabel_nodes=False,
                num_nodes=num_nodes
            )
            
            # Exclude the center node itself
            neighbors = [n for n in subset.tolist() if n != node_idx]
            
            if len(neighbors) > 0:
                # Sample neighbors if there are too many
                if len(neighbors) > sample_size:
                    sampled_neighbors = torch.tensor(
                        np.random.choice(neighbors, sample_size, replace=False)
                    )
                else:
                    sampled_neighbors = torch.tensor(neighbors)
                
                # Aggregate neighbor embeddings (mean pooling)
                neighbor_embs = node_embeddings[sampled_neighbors]
                multi_hop_embs[hop, node_idx] = neighbor_embs.mean(dim=0)
            else:
                # No neighbors at this hop: use the node's own embedding
                multi_hop_embs[hop, node_idx] = node_embeddings[node_idx]

print(f"\n✓ Multi-hop embeddings shape: {multi_hop_embs.shape}")
print(f"  Expected: [{num_hops}, {num_nodes}, {embedding_dim}]")
print(f"\nEmbedding structure:")
print(f"  - Hop 0: Entity itself")
print(f"  - Hop 1-{num_hops-1}: Aggregated k-hop neighbors")

                    STEP 3: Building Graph & Multi-hop Embeddings
Building graph structure from all triples...


                    STEP 3: Building Graph & Multi-hop Embeddings
Building graph structure from all triples...



✓ Graph structure:
  Nodes: 14,541
  Edges: 310,116


## 7. Create Conversation Data for Link Prediction

Format dataset splits as conversations: given (head, relation), predict tail.

In [ ]:
print("="*70)
print(" "*20 + "STEP 4: Creating Conversation Data")
print("="*70)

def format_relation(rel_uri):
    """Convert relation URI to readable text."""
    # Remove namespace and format
    rel = rel_uri.replace('/', ' ').replace('_', ' ').replace('.', ' ').strip()
    return rel if rel else rel_uri

# Create conversations from dataset splits
def create_conversations_from_split(split_data, split_name):
    conversations = []
    print(f"\nProcessing {split_name} split...")
    
    for item in tqdm(split_data, desc=f"Creating {split_name} conversations"):
        head_entity = item['head']
        tail_entity = item['tail']
        relation = item['relation']
        
        # Get entity IDs and texts (using rich descriptions)
        head_id = entity_to_id[head_entity]
        tail_id = entity_to_id[tail_entity]
        
        # Use entity names for conversation (shorter and clearer)
        head_name = entity_to_name.get(head_entity, head_entity.replace('/m/', '').replace('_', ' '))
        tail_name = entity_to_name.get(tail_entity, tail_entity.replace('/m/', '').replace('_', ' '))
        relation_text = format_relation(relation)
        
        # Format: Given head entity and relation, predict tail entity
        conversations.append({
            "id": [head_id],  # Use head entity for multi-hop embedding
            "conversations": [
                {
                    "from": "human",
                    "value": (
                        f"Given the entity '{head_name}' and relation '{relation_text}', "
                        f"predict the tail entity. Output ONLY the entity name, nothing else."
                    )
                },
                {
                    "from": "gpt",
                    "value": tail_name
                }
            ],
            "graph": 1,
            "triple": {
                "head": head_entity,
                "head_name": head_name,
                "relation": relation,
                "tail": tail_entity,
                "tail_name": tail_name,
                "head_id": head_id,
                "tail_id": tail_id
            }
        })
    
    return conversations

# Create conversations for each split
train_conversations = create_conversations_from_split(dataset['train'], 'train')
val_conversations = create_conversations_from_split(dataset['validation'], 'validation')
test_conversations = create_conversations_from_split(dataset['test'], 'test')

print(f"\n✓ Created conversations:")
print(f"  Train: {len(train_conversations):,}")
print(f"  Validation: {len(val_conversations):,}")
print(f"  Test: {len(test_conversations):,}")
print(f"  Total: {len(train_conversations) + len(val_conversations) + len(test_conversations):,}")

# Show relation distribution
print("\nRelation distribution (train):")
rel_counts = defaultdict(int)
for conv in train_conversations:
    rel_counts[conv['triple']['relation']] += 1

for rel, count in sorted(rel_counts.items(), key=lambda x: -x[1])[:10]:
    rel_text = format_relation(rel)
    print(f"  {rel_text[:50]}: {count:,}")
if len(rel_counts) > 10:
    print(f"  ... and {len(rel_counts) - 10} more relations")

## 8. Save All Files

Save conversations, embeddings, and mappings to `/kaggle/working/`.

In [ ]:
print("="*70)
print(" "*20 + "STEP 5: Saving Files")
print("="*70)

output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}\n")

# Save train conversations
train_path = output_dir / "fb15k237_train_link_data.jsonl"
with open(train_path, 'w', encoding='utf-8') as f:
    for conv in train_conversations:
        f.write(json.dumps(conv, ensure_ascii=False) + '\n')
train_size = os.path.getsize(train_path) / (1024**2)
print(f"✓ Saved: {train_path.name}")
print(f"  Samples: {len(train_conversations):,}")
print(f"  Size: {train_size:.2f} MB")

# Save validation conversations
val_path = output_dir / "fb15k237_val_link_data.jsonl"
with open(val_path, 'w', encoding='utf-8') as f:
    for conv in val_conversations:
        f.write(json.dumps(conv, ensure_ascii=False) + '\n')
val_size = os.path.getsize(val_path) / (1024**2)
print(f"\n✓ Saved: {val_path.name}")
print(f"  Samples: {len(val_conversations):,}")
print(f"  Size: {val_size:.2f} MB")

# Save test conversations
test_path = output_dir / "fb15k237_test_link_data.jsonl"
with open(test_path, 'w', encoding='utf-8') as f:
    for conv in test_conversations:
        f.write(json.dumps(conv, ensure_ascii=False) + '\n')
test_size = os.path.getsize(test_path) / (1024**2)
print(f"\n✓ Saved: {test_path.name}")
print(f"  Samples: {len(test_conversations):,}")
print(f"  Size: {test_size:.2f} MB")

# Save base node embeddings
node_emb_path = output_dir / "node_embeddings.pt"
torch.save(node_embeddings, node_emb_path)
node_emb_size = os.path.getsize(node_emb_path) / (1024**2)
print(f"\n✓ Saved: {node_emb_path.name}")
print(f"  Shape: {node_embeddings.shape}")
print(f"  Size: {node_emb_size:.2f} MB")

# Save multi-hop embeddings
multi_hop_path = output_dir / "multi_hop_graph_embedding.pt"
torch.save(multi_hop_embs, multi_hop_path)
multi_hop_size = os.path.getsize(multi_hop_path) / (1024**2)
print(f"\n✓ Saved: {multi_hop_path.name}")
print(f"  Shape: {multi_hop_embs.shape}")
print(f"  Size: {multi_hop_size:.2f} MB")

# Save comprehensive entity mappings including names and descriptions
mappings = {
    'entities': entities,
    'entity_texts': entity_texts,
    'entity_to_name': entity_to_name,
    'entity_to_desc': entity_to_desc,
    'entity_to_id': entity_to_id,
    'relations': relations,
    'relation_to_id': relation_to_id,
    'statistics': {
        'total_entities': len(entities),
        'entities_with_name_and_desc': entities_with_desc,
        'entities_with_name_only': entities_with_name,
        'entities_fallback': entities_fallback,
        'total_relations': len(relations)
    }
}
mappings_path = output_dir / "entity_relation_mappings.json"
with open(mappings_path, 'w', encoding='utf-8') as f:
    json.dump(mappings, f, indent=2, ensure_ascii=False)
mappings_size = os.path.getsize(mappings_path) / (1024**2)
print(f"\n✓ Saved: {mappings_path.name}")
print(f"  Entities: {len(entities):,}")
print(f"  Relations: {len(relations):,}")
print(f"  Size: {mappings_size:.2f} MB")

# Calculate total size
total_size = train_size + val_size + test_size + node_emb_size + multi_hop_size + mappings_size

print(f"\n{'='*70}")
print(f"✅ ALL FILES SAVED SUCCESSFULLY!")
print(f"{'='*70}")
print(f"Total size: {total_size:.2f} MB")
print(f"Location: {output_dir}")

## 9. Summary and Download Instructions

In [ ]:
print("="*70)
print(" "*20 + "✅ PREPARATION COMPLETE!")
print("="*70)

print("\n📊 Summary:")
print(f"  • Dataset: FB15k-237 Knowledge Graph (KGraph)")
print(f"  • Task: Link Prediction")
print(f"  • Total entities: {len(entities):,}")
print(f"  • Total relations: {len(relations):,}")
print(f"  • Train samples: {len(train_conversations):,}")
print(f"  • Val samples: {len(val_conversations):,}")
print(f"  • Test samples: {len(test_conversations):,}")
print(f"  • Multi-hop embeddings: {multi_hop_embs.shape}")
print(f"  • Total output size: {total_size:.2f} MB")

print("\n📥 Download Instructions:")
print("  1. Click 'Output' tab on the right sidebar")
print("  2. Click 'Download All' to get fb15k237_gwm_data.zip")
print("  3. Or download files individually:")

for file in sorted(output_dir.glob("*")):
    size_mb = os.path.getsize(file) / (1024**2)
    print(f"     • {file.name} ({size_mb:.2f} MB)")

print("\n🚀 Next Steps:")
print("  1. Download the files from Kaggle")
print("  2. Upload to your training environment")
print("  3. Update file paths in GWM training script")
print("  4. Train for link prediction:")
print("     python train.py \\")
print("       --train_jsonl fb15k237_train_link_data.jsonl \\")
print("       --val_jsonl fb15k237_val_link_data.jsonl \\")
print("       --test_jsonl fb15k237_test_link_data.jsonl \\")
print("       --embedding_path multi_hop_graph_embedding.pt")
print("\n  💡 Dataset uses official FB15k-237 splits:")
print("     Train/Val/Test splits are already properly balanced!")
print("     Entity URIs converted to readable text for LLM training.")

print("\n" + "="*70)

# Show example conversation
print("\n📝 Example Conversation:")
print("-"*70)
example = json.dumps(train_conversations[0], indent=2)
if len(example) > 600:
    print(example[:600] + "\n...")
else:
    print(example)

print("\n" + "="*70)
print("Notebook completed successfully! 🎉")
print("="*70)

## Optional: Verify Files

In [ ]:
# Quick verification
print("Verifying saved files...\n")

# Load and check embeddings
loaded_embs = torch.load(multi_hop_path)
print(f"✓ Multi-hop embeddings: {loaded_embs.shape}")
assert loaded_embs.shape == multi_hop_embs.shape

# Load and check conversations
with open(train_path, 'r') as f:
    train_lines = f.readlines()
print(f"✓ Train conversations: {len(train_lines)} lines")
assert len(train_lines) == len(train_conversations)

with open(val_path, 'r') as f:
    val_lines = f.readlines()
print(f"✓ Val conversations: {len(val_lines)} lines")
assert len(val_lines) == len(val_conversations)

with open(test_path, 'r') as f:
    test_lines = f.readlines()
print(f"✓ Test conversations: {len(test_lines)} lines")
assert len(test_lines) == len(test_conversations)

# Load and check mappings
with open(mappings_path, 'r') as f:
    loaded_mappings = json.load(f)
print(f"✓ Mappings loaded:")
print(f"  Entities: {len(loaded_mappings['entities'])} ")
print(f"  Relations: {len(loaded_mappings['relations'])}")
assert len(loaded_mappings['entities']) == len(entities)
assert len(loaded_mappings['relations']) == len(relations)

print("\n✅ All files verified successfully!")